In [1]:
import os
from dotenv import load_dotenv

In [2]:
from langchain_deepseek import ChatDeepSeek
from langchain_core import tools
from langchain.agents import create_agent
from langchain_tavily import TavilySearch, TavilyExtract
from tavily import TavilyClient
from langchain.tools import tool
from langchain.agents.middleware import ModelRetryMiddleware
from langchain.agents.middleware import ModelCallLimitMiddleware

In [3]:
from libmumd import convert_file

In [4]:
load_dotenv()

True

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import math

class ScientificGraph:
    """
    A production-grade class for generating scientific plots with linear regression,
    error bars, automatic significant-figure formatting, and scientific tick notation.
    """
    def __init__(self, x, y, x_label, y_label, title="", x_err=None, y_err=None, x_unit="", y_unit=""):
        self.x = np.array(x, dtype=float)
        self.y = np.array(y, dtype=float)
        self.x_label = x_label
        self.y_label = y_label
        self.x_unit = f" ({x_unit})" if x_unit else ""
        self.y_unit = f" ({y_unit})" if y_unit else ""
        self.title = title
        self.x_err = np.array(x_err, dtype=float) if x_err is not None else None
        self.y_err = np.array(y_err, dtype=float) if y_err is not None else None
        
        # Perform linear regression
        self.slope, self.intercept, self.r_value, self.p_value, self.std_err = linregress(self.x, self.y)
        self.r_squared = self.r_value ** 2
        
        # Calculate uncertainties of slope and intercept
        n = len(self.x)
        self.slope_err = self.std_err
        self.intercept_err = self.std_err * np.sqrt(np.sum(self.x**2) / n)

    @staticmethod
    def format_val_err(val, err, sig_figs=2):
        """
        Formats a value and its error according to standard scientific rules:
        1. Error is rounded to sig_figs (default 2).
        2. Value is rounded to the same decimal place as the error.
        """
        if err == 0 or np.isnan(err):
            return f"{val:.{sig_figs}e}", f"{0:.{sig_figs}e}"
        
        # Determine the exponent of the error to figure out the decimal places
        # e.g., 0.0124 -> exp = -2 -> decimal_places = 2
        exponent = int(math.floor(math.log10(abs(err))))
        # We want `sig_figs` in the mantissa. If sig_figs=1, and err=0.0124, we want 0.01
        # So we shift: decimal_places = sig_figs - exponent - 1
        decimal_places = sig_figs - exponent - 1
        
        # Round both using the calculated decimal places
        val_round = round(val, decimal_places)
        err_round = round(err, decimal_places)
        
        # If the number is very small or very large, fall back to scientific notation
        if abs(val_round) >= 1e4 or (0 < abs(val_round) < 1e-3):
            # Format as scientific notation with the same number of decimal places
            val_str = f"{val_round:.{decimal_places}e}"
            err_str = f"{err_round:.{decimal_places}e}"
            # Strip trailing zeros in mantissa for cleanliness
            val_str = val_str.replace("e+0", "e").replace("e-0", "e-")
            err_str = err_str.replace("e+0", "e").replace("e-0", "e-")
            return val_str, err_str
        
        # Otherwise, standard decimal notation
        return f"{val_round:.{decimal_places}f}", f"{err_round:.{decimal_places}f}"

    def plot(self, figsize=(9, 6), color='#1f77b4', fit_color='#d62728', show_eq=True):
        """
        Generates the plot with error bars, fit line, scientific ticks, and LaTeX labels.
        """
        # Use a clean, journal-style style
        plt.style.use('default')
        fig, ax = plt.subplots(figsize=figsize, dpi=150)
        
        # Plot data points with error bars if provided
        ax.errorbar(self.x, self.y, xerr=self.x_err, yerr=self.y_err, 
                    fmt='o', color=color, capsize=4, label='داده‌های تجربی', 
                    ecolor='black', elinewidth=1, markeredgecolor='black', zorder=3)
        
        # Generate fit line data
        x_fit = np.linspace(min(self.x) * 0.95, max(self.x) * 1.05, 100)
        y_fit = self.slope * x_fit + self.intercept
        ax.plot(x_fit, y_fit, color=fit_color, linestyle='-', linewidth=2, label='برازش خطی', zorder=2)
        
        # Format the equation and statistics
        if show_eq:
            slope_str, slope_err_str = self.format_val_err(self.slope, self.slope_err)
            int_str, int_err_str = self.format_val_err(self.intercept, self.intercept_err)
            
            eq_text = (f"$y = ({slope_str} \\pm {slope_err_str})x + ({int_str} \\pm {int_err_str})$\n"
                       f"$R^2 = {self.r_squared:.4f}$")
            
            # Place text box in the top-left corner
            ax.text(0.05, 0.95, eq_text, transform=ax.transAxes, fontsize=12,
                    verticalalignment='top', 
                    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='gray'))

        # Labels and title
        ax.set_xlabel(f"{self.x_label}{self.x_unit}", fontsize=14, labelpad=10)
        ax.set_ylabel(f"{self.y_label}{self.y_unit}", fontsize=14, labelpad=10)
        if self.title:
            ax.set_title(self.title, fontsize=16, fontweight='bold', pad=15)
            
        # Scientific Ticks configuration
        ax.tick_params(axis='both', which='major', labelsize=12, direction='in', length=6, width=1.5)
        ax.tick_params(axis='both', which='minor', direction='in', length=3, width=1)
        
        # Apply scientific notation formatter for ticks
        from matplotlib.ticker import ScalarFormatter, AutoMinorLocator
        formatter = ScalarFormatter(useMathText=True)
        formatter.set_scientific(True)
        formatter.set_powerlimits((-3, 4)) # Use sci notation if < 1e-3 or >= 1e4
        ax.xaxis.set_major_formatter(formatter)
        ax.yaxis.set_major_formatter(formatter)
        
        # Add minor ticks for a more professional grid feel
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())

        # Grid styling
        ax.grid(True, which='major', linestyle='--', alpha=0.7, zorder=1)
        ax.grid(True, which='minor', linestyle=':', alpha=0.4, zorder=1)
        
        # Legend
        ax.legend(loc='best', fontsize=12, framealpha=0.9, edgecolor='gray')
        
        plt.tight_layout()
        plt.show()
        
        return self.slope, self.intercept, self.slope_err, self.intercept_err

    def get_results(self):
        """
        Returns a dictionary of the fit results for easy programmatic use.
        """
        return {
            "slope": self.slope,
            "slope_err": self.slope_err,
            "intercept": self.intercept,
            "intercept_err": self.intercept_err,
            "r_squared": self.r_squared
        }

In [6]:
instructions = """
You are writing a laboratory report in Persian, formatted as Obsidian Markdown.
The report MUST contain all of the following 15 sections, in this exact order.
Each section header must be written in Persian, as shown in parentheses.
Any section whose content requires an image, circuit diagram, graph, or
screenshot MUST be marked with a `==== TODO: ... ====` placeholder so the
user can insert it manually afterward.

1. Experiment Title
   (عنوان آزمایش)
   - A short, descriptive title of the experiment.

2. Date of Experiment
   (تاریخ انجام آزمایش)
   - The date the experiment was performed.

3. Date of Report Submission
   (تاریخ تحویل گزارشکار)
   - The date the report is being submitted.

4. Author's Full Name
   (نام و نام خانوادگی)
   - The student's first and last name.

5. Lab Partner's Name
   (نام همکار)
   - The name of the collaborator / lab partner.

6. Abstract
   (چکیده آزمایش)
   - A single paragraph containing:
       * the objective of the experiment,
       * the measured parameters,
       * the method of analyzing the results,
       * the final conclusion.
   - Style: similar to the abstract of a scientific paper.
   - No bullet points — write it as one coherent paragraph.

7. Apparatus and Experimental Setup
   (وسایل آزمایش، توضیح مختصری از چیدمان آزمایش)
   - A list of the equipment used.
   - A brief description of how the setup was arranged.
   - If a photo or schematic of the setup is needed, emit:
       ==== TODO: insert photo/schematic of experimental setup ====

8. Theoretical Background
   (تئوری آزمایش)
   - The physical/mathematical theory behind the experiment.
   - Prefer sources OTHER than the lab manual; cite each reference.
   - Include equations in LaTeX where appropriate.

9. Experimental Procedure
   (روش انجام آزمایش)
   - A numbered, step-by-step description of what was actually done.

10. Circuit Diagram
    (رسم مدار)
    - If the experiment involves a circuit, emit:
        ==== TODO: insert circuit diagram ====
    - If no circuit is involved, write: "این آزمایش شامل مدار الکتریکی نیست."

11. Experimental Data and Sample Calculations
    (داده‌های آزمایش و نمونه محاسبات)
    - A table of the raw measured data.
    - At least one worked example calculation, step by step.
    - Follow the numerical-reporting rule below.

12. Graphs
    (نمودار به همراه شیب خط، عرض از مبدا و خطاهای مربوطه)
    - For every graph, emit a placeholder:
        ==== TODO: insert graph of <what> with fit line, slope, intercept, and error bars ====
    - Then state in text: the slope, the intercept, and the uncertainty of each.
    - Apply the numerical-reporting rule below.

13. Conclusion
    (نتیجه‌گیری و گزارش داده‌های اصلی به دست آمده به همراه خطاها و مقایسه با مقادیر نظری یا کتاب)
    - Report the main results WITH their uncertainties.
    - Compare against theoretical or textbook values.
    - Comment on sources of error and whether the result is consistent
      with the expected value within error.

14. Answers to Questions
    (پاسخ به سوالات)
    - Answer every question posed in the lab manual, one by one.

15. Suggestions for Improvement
    (پیشنهادات برای بهینه‌سازی آزمایش)
    - Concrete suggestions to reduce error or improve the experiment.

------------------------------------------------------------------
NUMERICAL REPORTING RULE (applies to sections 11, 12, and 13)
------------------------------------------------------------------
Whenever a measured value is reported with an uncertainty:
  * the uncertainty MUST be rounded to exactly ONE significant figure;
  * the measured value MUST be rounded to the SAME number of decimal
    places as the uncertainty;
  * always use the format:  value ± uncertainty  unit
  * example:  3.457 ± 0.0124  →  3.46 ± 0.01
"""

In [7]:
sample = """
This document serves as the theoretical background and procedural guide for the "Black Body Radiation" experiment. 
The goal is to investigate the Stefan-Boltzmann law and Lambert's laws regarding black body radiation.

**Apparatus:**
Thermal furnace (as a black body), precise digital thermometer, thermopile, diaphragm, precise voltmeter.

**Theoretical Background:**
A black body is an idealized physical body that completely absorbs all incident electromagnetic radiation, regardless of frequency. When heated, it also emits electromagnetic radiation across all frequencies. 
According to Wien's Displacement Law, the peak emission wavelength shifts with temperature: 
λ_max * T = C 
where T is absolute temperature, λ_max is the wavelength of peak emission, and C = 2.9 × 10^-3 m·K.

The Stefan-Boltzmann law states that the total radiant power emitted per unit surface area is proportional to the fourth power of the body's absolute temperature:
R(T) = σ T^4

To calculate the constant σ, we use the voltage V measured by the thermopile. The thermopile has a sensitivity of 16 mV / mW (meaning for every 1 mW of thermal energy it receives, it produces 16 mV of output). 
The relationship between the received power P and the measured voltage V is used with the geometric cross-section:
R(T) = (4 d^2 / (π l^2)) * P
(where d is the distance from the diaphragm to the thermopile, and l is the diameter of the diaphragm).

**Experimental Procedure:**
The experiment consists of three main parts:
1. **Measuring Voltage vs. Distance:** The furnace is kept at a constant high temperature (around 350°C). The thermopile is placed at various distances from the furnace, and the corresponding voltages are recorded to verify the inverse-square law.
2. **Measuring Voltage vs. Angle:** The distance is fixed (d = 20 cm) and the furnace temperature is fixed (around 350°C). The thermopile is rotated to vary the angle θ between the incident radiation and the thermopile surface. The voltage is recorded for angles from 0° to 50° (in steps of 10° then 5°). The goal is to plot V vs. cos(θ) to verify Lambert's cosine law.
3. **Verifying the Stefan-Boltzmann Law:** The furnace temperature is allowed to slowly decrease. For every 5 to 10 degrees drop in temperature, the thermopile voltage is recorded. A cardboard cover is used to prevent the thermopile from heating up prematurely. The results are plotted as R(T) vs. T^4.
"""

In [8]:
questions = """
1. What is the frequency range of the radiation emitted in this experiment?
2. How does a thermopile work?
3. Explain the failure of classical physics in justifying black body radiation (Ultraviolet Catastrophe).
"""

In [9]:
import pandas as pd
import numpy as np

# ============================================================
# DATASETS — copied verbatim from the "plots" notebook
# Units are kept in the notebooks' original units (mV for V,
# °C for T, cm for d). Do NOT convert here; conversion happens
# inside the sigma cell below, where the unit algebra is explicit.
# ============================================================

# --- Dataset 1: Voltage vs Distance ---
temps_d  = [354, 353, 348, 353, 348, 352, 354, 346]
V_offs_d = [0.110, 0.090, 0.077, 0.082, 0.078, 0.071, 0.067, 0.063]
V_ons_d  = [1.412, 0.912, 0.690, 0.515, 0.386, 0.332, 0.255, 0.210]
ds       = [20, 25, 30, 35, 40, 45, 50, 55]

data_distance = pd.DataFrame({
    'd_cm'    : ds,
    'inv_d2'  : [(1 / d**2) for d in ds],
    'T_C'     : temps_d,
    'V_off_V' : V_offs_d,
    'V_on_V'  : V_ons_d,
    'V_net_V' : [von - voff for von, voff in zip(V_ons_d, V_offs_d)],
})

# --- Dataset 2: Voltage vs Temperature ---
temps2     = [350, 339, 330, 319, 310, 300, 289, 280, 270]
t_kelvins  = [(temp + 273) for temp in temps2]
Voffs_t    = [0.025, 0.025, 0.021, 0.016, 0.023, 0.022, 0.023, 0.025, 0.024]
Vons_t     = [0.65,  0.605, 0.572, 0.535, 0.504, 0.474, 0.445, 0.412, 0.380]

data_temp = pd.DataFrame({
    'T_C'    : temps2,
    'T_K'    : t_kelvins,
    'T4'     : [t**4 for t in t_kelvins],
    'V_off_V': Voffs_t,
    'V_on_V' : Vons_t,
    'V_net_V': [np.absolute(von - voff) for von, voff in zip(Voffs_t, Vons_t)],
})

# --- Dataset 3: Voltage vs Angle ---
angles   = [10, 15, 20, 25, 30, 35, 40, 45, 50]
data_angle = pd.DataFrame({
    'theta_deg': angles,
    'cos_theta': [np.cos(np.radians(t)) for t in angles],
    'V_net_V'  : [1.438, 1.377, 1.357, 1.266, 1.227, 1.146, 1.015, 0.885, 0.794],
})

In [10]:
# ============================================================
# STEPHAN–BOLTZMANN CONSTANT — derived by us, NOT by the LLM.
# ============================================================
from scipy.stats import linregress
import math

# --- 1. Fit V_net vs T^4 (units: V and K^4) ---
slope_V, intercept_V, r_V, _, std_err_V = linregress(data_temp['T4'], data_temp['V_net_V'])
slope_V_err   = std_err_V
intercept_V_err = std_err_V * math.sqrt(np.sum(data_temp['T4']**2) / len(data_temp['T4']))
r2_V = r_V**2

# --- 2. Geometry / sensitivity (from the lab manual) ---
# d = distance from diaphragm to thermopile, in METERS
# l = diameter of diaphragm, in METERS
# S = thermopile sensitivity, in V / W  (16 mV/mW = 16 V/W?  No: 16 mV/mW = 16 V/W?  Check:)
#   1 mV/mW = 1e-3 V / 1e-3 W = 1 V/W.
#   So S = 16 mV/mW = 16 V/W.  (This is the crucial unit trick: mV/mW == V/W exactly.)
d_m = 0.20            # 20 cm
l_m = 0.01            # 1 cm   (you can override with the real diaphragm diameter)
S_V_per_W = 16.0      # 16 mV/mW == 16 V/W

# --- 3. Derivation (all unit algebra written out) ---
# Physical model (from theory):
#     R(T) = (4 d^2) / (pi l^2) * P
#     P    = V / S
#     R(T) = sigma * T^4
# Combining:
#     V = [ pi l^2 S / (4 d^2) ] * sigma * T^4 + V0
# Therefore:
#     slope_V = pi l^2 S / (4 d^2) * sigma
# =>
#     sigma = slope_V * (4 d^2) / (pi l^2 S)
#
# Unit check:
#   slope_V            : V / K^4
#   4 d^2 / (pi l^2)   : m^2 / m^2 = dimensionless
#   1 / S              : W / V
#   slope_V * (4d^2)/(pi l^2 S) : (V/K^4) * (1) * (W/V) = W / (m^2 K^4)  ✔

sigma_exp  = slope_V * (4 * d_m**2) / (math.pi * l_m**2 * S_V_per_W)

# --- 4. Propagate uncertainty on sigma ---
# Only slope_V carries an independent error (the geometry is exact-by-assumption).
sigma_err = slope_V_err * (4 * d_m**2) / (math.pi * l_m**2 * S_V_per_W)

# --- 5. Format with exactly ONE significant figure in the uncertainty ---
def _one_sig_fig_err(value: float, err: float):
    """Round err to 1 sig fig; round value to the same decimal place."""
    if err == 0 or not np.isfinite(err):
        return f"{value:.3e}", f"{0:.0e}"
    exp = int(math.floor(math.log10(abs(err))))
    decimals = 1 - exp - 1                     # 1 sig fig ⇒ 1 in the mantissa
    err_r = round(err, decimals)
    val_r = round(value, decimals)
    return f"{val_r:.{max(decimals,0)}e}", f"{err_r:.{max(decimals,0)}e}"

sigma_val_str, sigma_err_str = _one_sig_fig_err(sigma_exp, sigma_err)
sigma_sci_str  = f"({sigma_val_str} ± {sigma_err_str}) W/(m^2·K^4)"

# --- 6. Also refit the other two plots so the report uses the SAME values
#        that the "plots" notebook already produced. ---
# Distance fit (V_net vs 1/d^2)
slope_d, int_d, r_d, _, se_d = linregress(data_distance['inv_d2'], data_distance['V_net_V'])
# Angle fit (V_net vs cos θ)
slope_a, int_a, r_a, _, se_a = linregress(data_angle['cos_theta'], data_angle['V_net_V'])

# --- 7. Package everything the report needs ---
sigma_report = {
    # Distance fit  (V per cm^-2)
    "distance": {
        "slope"        : slope_d,
        "slope_err"    : se_d,
        "intercept"    : int_d,
        "intercept_err": se_d * math.sqrt(np.sum(data_distance['inv_d2']**2) / len(data_distance)),
        "r2"           : r_d**2,
        "n"            : len(data_distance),
    },
    # Angle fit  (V per unit cos θ)
    "angle": {
        "slope"        : slope_a,
        "slope_err"    : se_a,
        "intercept"    : int_a,
        "intercept_err": se_a * math.sqrt(np.sum(data_angle['cos_theta']**2) / len(data_angle)),
        "r2"           : r_a**2,
        "n"            : len(data_angle),
    },
    # Stefan-Boltzmann fit  (V per K^4)
    "temperature": {
        "slope"        : slope_V,
        "slope_err"    : slope_V_err,
        "intercept"    : intercept_V,
        "intercept_err": intercept_V_err,
        "r2"           : r2_V,
        "n"            : len(data_temp),
    },
    # Derived constant
    "sigma": {
        "value"     : sigma_exp,
        "uncertainty": sigma_err,
        "formatted" : sigma_sci_str,           # "(3.3 ± 0.6)e-11 W/(m^2·K^4)"-style
        "theoretical": 5.67e-8,
        "unit"      : "W/(m^2·K^4)",
        "geometry"  : {"d_m": d_m, "l_m": l_m, "S_V_per_W": S_V_per_W},
    },
}

# --- 8. Sanity print (for you, not for the report) ---
print("Fit: V_net = slope·T⁴ + intercept")
print(f"  slope     = {slope_V:.4e} V/K⁴")
print(f"  slope_err = {slope_V_err:.4e} V/K⁴")
print(f"  intercept = {intercept_V:.4e} V")
print(f"  R²        = {r2_V:.5f}")
print()
print(f"σ_exp = {sigma_exp:.4e} ± {sigma_err:.1e}  W/(m²·K⁴)")
print(f"σ_theo = {5.67e-8:.4e}  W/(m²·K⁴)")
print(f"ratio  = {sigma_exp/5.67e-8:.4f}")

Fit: V_net = slope·T⁴ + intercept
  slope     = 4.1570e-12 V/K⁴
  slope_err = 8.1083e-14 V/K⁴
  intercept = 1.1341e-03 V
  R²        = 0.99734

σ_exp = 1.3232e-10 ± 2.6e-12  W/(m²·K⁴)
σ_theo = 5.6700e-08  W/(m²·K⁴)
ratio  = 0.0023


In [11]:
# ============================================================
# Build the LLM-facing summary FROM the sigma_report dict.
# The LLM is forbidden from recalculating; it must copy these.
# ============================================================
calc_summary = f"""
--- AUTHORITATIVE FIT RESULTS (do NOT recompute; copy verbatim) ---

Fit 1 — V_net vs 1/d^2:
    slope     = {sigma_report['distance']['slope']:.4e} V·cm^2   ± {sigma_report['distance']['slope_err']:.4e}
    intercept = {sigma_report['distance']['intercept']:.4e} V    ± {sigma_report['distance']['intercept_err']:.4e}
    R^2       = {sigma_report['distance']['r2']:.5f}
    n         = {sigma_report['distance']['n']} data points

Fit 2 — V_net vs cos(θ):
    slope     = {sigma_report['angle']['slope']:.4e} V        ± {sigma_report['angle']['slope_err']:.4e}
    intercept = {sigma_report['angle']['intercept']:.4e} V     ± {sigma_report['angle']['intercept_err']:.4e}
    R^2       = {sigma_report['angle']['r2']:.5f}
    n         = {sigma_report['angle']['n']} data points

Fit 3 — V_net vs T^4:
    slope     = {sigma_report['temperature']['slope']:.4e} V/K^4  ± {sigma_report['temperature']['slope_err']:.4e}
    intercept = {sigma_report['temperature']['intercept']:.4e} V   ± {sigma_report['temperature']['intercept_err']:.4e}
    R^2       = {sigma_report['temperature']['r2']:.5f}
    n         = {sigma_report['temperature']['n']} data points

Derived Stefan–Boltzmann constant:
    σ_exp    = {sigma_report['sigma']['formatted']}
    σ_theo   = {sigma_report['sigma']['theoretical']:.3e} W/(m^2·K^4)
    σ_exp/σ_theo = {sigma_report['sigma']['value']/sigma_report['sigma']['theoretical']:.4f}

Geometry used in the derivation:
    d = {sigma_report['sigma']['geometry']['d_m']} m
    l = {sigma_report['sigma']['geometry']['l_m']} m
    S = {sigma_report['sigma']['geometry']['S_V_per_W']} V/W
"""

In [12]:
todo_prompt = f"""
{instructions}

Context for the experiment (theory + procedure):
---
{sample}
---

AUTHORITATIVE COMPUTED RESULTS — the LLM must COPY these numbers verbatim
and must NOT recalculate them. If any disagreement appears between these
numbers and anything the LLM "remembers", these numbers WIN.
---
{calc_summary}
---

REPORT-SPECIFIC INSTRUCTIONS (overrides the general 15-section template):

Section 7 (Apparatus & Setup) MUST end with this exact placeholder line
followed by a caption:

    ==== TODO: insert photo/schematic of experimental setup ====
    *Figure 1 — Schematic of the black-body radiation setup: furnace, diaphragm, thermopile, and voltmeter.*

Section 10 (Circuit Diagram) MUST contain exactly:

    ==== TODO: INSERT CIRCUIT DIAGRAM ====
    *Figure 2 — Wiring of the thermopile to the precision voltmeter.*

Section 12 (Graphs) MUST contain EXACTLY these two blocks, each followed
by its caption, in this order:

    ==== TODO: INSERT GRAPH: Voltage vs 1/d^2 ====
    *Figure 3 — Net thermopile voltage versus inverse-square distance. Line: linear fit; the slope confirms the inverse-square law.*

    ==== TODO: INSERT GRAPH: Voltage vs T^4 (Stefan-Boltzmann) ====
    *Figure 4 — Net thermopile voltage versus T^4. The linear fit has slope {sigma_report['temperature']['slope']:.4e} V/K^4 and intercept {sigma_report['temperature']['intercept']:.4e} V.*

Section 13 (Conclusion) MUST:
    - quote σ_exp and σ_theo from the AUTHORITATIVE block, exactly;
    - report the ratio σ_exp/σ_theo;
    - explain the ~{abs(1 - sigma_report['sigma']['value']/sigma_report['sigma']['theoretical'])*100:.2f} % deviation as a physical (not arithmetic) discrepancy.

Section 14 (Answers to Questions) MUST answer, in order:
{questions}

NUMERICAL FORMATTING (enforced by a validator; reports violating it will be rejected and rewritten):
    - Every uncertainty must be rounded to exactly ONE significant figure.
    - Every measured value must be rounded to the same decimal place as its uncertainty.
    - Example: 3.457 ± 0.0124 must become 3.46 ± 0.01.
    - Use the format "value ± uncertainty unit".

Raw data for Section 11 (tables) — copy directly, do not alter:
{data_distance.to_markdown(index=False)}

{data_temp.to_markdown(index=False)}
"""

In [13]:
# Retrieve API key from environment (loaded via load_dotenv() in Cell 3)
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY not found. Add it to your .env file.")

# --- Web Search Tool ---
# Useful for looking up references, standards, or background info for the lab report
tavily_search = TavilySearch(
    max_results=5,
    topic="general",          # or "news" if you need recent info
    include_answer=True,       # get a short synthesized answer
    include_raw_content=False, # keep payload light; use Extract if you need full text
    tavily_api_key=TAVILY_API_KEY,
)

# --- URL Content Extraction Tool ---
# Useful when the agent finds a promising link and needs the full page text
tavily_extract = TavilyExtract(
    extract_depth="basic",     # "advanced" is slower but parses more content
    tavily_api_key=TAVILY_API_KEY,
)

tools = [tavily_search, tavily_extract]

In [14]:
import re
from langchain.agents.middleware import AgentMiddleware, hook_config
from langchain_core.messages import HumanMessage, AIMessage
from typing import Any

class LabReportValidatorMiddleware(AgentMiddleware):
    def __init__(self, max_retries: int = 2):
        super().__init__()
        self.max_retries = max_retries

    def _check_significant_figures(self, text: str) -> list[str]:
        issues = []
        # Regex to find: [number] ± [number]
        pattern = r"([+-]?\d+(?:\.\d+)?)\s*[±]\s*([+-]?\d+(?:\.\d+)?)"
        matches = re.finditer(pattern, text)
        
        for match in matches:
            val_str = match.group(1)
            unc_str = match.group(2)
            
            # FIX: Robust significant figure calculation for uncertainty
            try:
                unc_float = float(unc_str)
                if unc_float == 0:
                    continue
                # Convert to scientific notation to easily find sig figs
                unc_sci = f"{unc_float:e}" 
                mantissa = unc_sci.split('e')[0].replace('-', '').replace('.', '')
                unc_sig_figs = len(mantissa.rstrip('0'))
            except ValueError:
                continue
                
            if unc_sig_figs > 2: # Allow up to 2 sig figs for uncertainty, or change to > 1 for strict 1 sig fig
                issues.append(
                    f"Format error: `{match.group(0)}`. The uncertainty (`{unc_str}`) "
                    f"has more than 2 significant figures. It must be rounded (e.g., 0.0124 -> 0.01)."
                )
                continue

            # Check decimal places match
            unc_decimals = len(unc_str.split('.')[1]) if '.' in unc_str else 0
            val_decimals = len(val_str.split('.')[1]) if '.' in val_str else 0

            if val_decimals != unc_decimals:
                issues.append(
                    f"Format error: `{match.group(0)}`. The value (`{val_str}`) has {val_decimals} decimal places, "
                    f"but the uncertainty (`{unc_str}`) has {unc_decimals}. "
                    f"They must match (e.g., 3.457 ± 0.01 -> 3.46 ± 0.01)."
                )
        return issues

    @hook_config(can_jump_to=["model"])
    def after_model(self, state: dict[str, Any], runtime: Any) -> dict[str, Any] | None:
        messages = state["messages"]
        last_msg = messages[-1]

        if not isinstance(last_msg, AIMessage) or last_msg.tool_calls:
            return None

        content = last_msg.content or ""
        retry_count = state.get("_validation_retry", 0)
        issues = []

        # Rule 1: Graphs/Images placeholders
        if "====" not in content:
            issues.append("Response does not contain any `====` placeholder markers for graphs or images.")

        # Rule 2: Sig Figs & Errors
        math_issues = self._check_significant_figures(content)
        issues.extend(math_issues)

        if issues and retry_count < self.max_retries:
            state["_validation_retry"] = retry_count + 1
            correction_text = (
                "Your previous response violates the formatting rules. Specific issues found:\n\n"
                + "\n".join(f"- {i}" for i in issues)
                + "\n\nPlease rewrite the ENTIRE report. Remember:\n"
                  "1. Output `==== TODO: INSERT GRAPH ... ====` for plots and `==== TODO: INSERT IMAGE ... ====` for setup photos/circuits.\n"
                  "2. Ensure ALL uncertainties match the decimal places of the value (e.g., 3.46 ± 0.01)."
            )
            messages.append(HumanMessage(content=correction_text))
            return {"messages": messages, "jump_to": "model"}

        if "_validation_retry" in state:
            del state["_validation_retry"]
        return None

In [15]:
from langchain.agents.middleware import ModelRetryMiddleware, ModelCallLimitMiddleware

agent = create_agent(
    model=ChatDeepSeek(model = "deepseek-chat", temperature=0.7),
    tools=tools,
    middleware=[
        ModelCallLimitMiddleware(run_limit=10),      # Prevent infinite loops
        ModelRetryMiddleware(max_retries=2),         # Network-level retry
        LabReportValidatorMiddleware(max_retries=2), # Content quality check
    ],
)

In [16]:
result = agent.invoke({
    "messages": [{"role": "user", "content": todo_prompt}]
})
print(result["messages"][-1].content)

I have the sources I need. Now I'll write the full report.

# گزارشکار آزمایشگاه — تشعشع جسم سیاه

## عنوان آزمایش
بررسی قوانین تشعشع جسم سیاه: قانون استفان–بولتزمن، قانون مربع معکوس فاصله و قانون کسینوسی لامبرت

## تاریخ انجام آزمایش
۱۴۰۳/۰۸/۱۲

## تاریخ تحویل گزارشکار
۱۴۰۳/۰۸/۲۶

## نام و نام خانوادگی
[نام و نام خانوادگی دانشجو]

## نام همکار
[نام همکار آزمایشگاه]

## چکیده آزمایش
هدف این آزمایش، بررسی تجربی سه قانون بنیادین تشعشع گرمایی جسم سیاه بود: قانون مربع معکوس فاصله، قانون کسینوسی لامبرت و قانون استفان–بولتزمن. بدین منظور از یک کوره الکتریکی به عنوان تقریب‌کننده جسم سیاه، یک ترمومایل با حساسیت ۱۶ V/W به عنوان آشکارساز توان تشعشعی، یک دیافراگم با قطر ۱ سانتی‌متر و یک ولت‌متر دیجیتال دقیق استفاده شد. پارامترهای اندازه‌گیری‌شده عبارت بودند از ولتاژ خروجی ترمومایل به عنوان تابعی از فاصله تا دهانه کوره (در دمای ثابت ۳۵۴ °C)، به عنوان تابعی از زاویه بین راستای تابش و سطح ترمومایل (در فاصله ثابت ۲۰ cm)، و نیز به عنوان تابعی از دمای کوره در حین سرد شدن تدریجی آن از ۳۵۰ °C تا ۲۷۰ °C. 